In [1]:
!git clone https://github.com/Kedar-V/InnerLight-RLHF_MentalHealthChatbot.git

Cloning into 'InnerLight-RLHF_MentalHealthChatbot'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), done.


In [ ]:
import pandas as pd
import kagglehub
from datasets import load_dataset

In [ ]:
path = kagglehub.dataset_download("birdy654/human-and-llm-mental-health-conversations")
print("Path to dataset files:", path)

dataset1 = path + '/dataset.csv'

Using Colab cache for faster access to the 'human-and-llm-mental-health-conversations' dataset.
Path to dataset files: /kaggle/input/human-and-llm-mental-health-conversations


In [ ]:
dataset1 = path + '/dataset.csv'
df_a = pd.read_csv(dataset1)
df_a = df_a.rename(columns={
    "LLM": "instruction",
    "Context": "input",
    "Response": "output"
})
df_a = df_a[["instruction", "input", "output"]]

GENERIC_INSTRUCTION = (
    "You are a helpful mental health counselling assistant. "
    "Please provide a safe, empathetic, and supportive answer to the user's input."
)

df_a["instruction"] = GENERIC_INSTRUCTION
print("Dataset A shape:", df_a.shape)


df_b = load_dataset("ShenLab/MentalChat16K")
df_b = df_b["train"].to_pandas()
df_b = df_b.rename(columns={
    "Context": "input",
    "Response": "output"
})

# ------------------------
# 5. CONCATENATE
# ------------------------

df_final = pd.concat([df_a, df_b], ignore_index=True)


# ------------------------
# 6. REMOVE DUPLICATES
# ------------------------

df_final = df_final.drop_duplicates(subset=["input", "output"])


# ------------------------
# 7. SHUFFLE
# ------------------------

df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)


# ------------------------
# 8. SAVE AS JSONL (FINETUNE READY)
# ------------------------

# df_final.to_json("final_mental_health_dataset.jsonl", orient="records", lines=True)
df_final.to_csv("mental_health_teacher_data.csv", index=False)


Dataset A shape: (3507, 3)


In [ ]:
pip install torch transformers peft pandas accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.6 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes

Found existing installation: bitsandbytes 0.48.2
Uninstalling bitsandbytes-0.48.2:
  Successfully uninstalled bitsandbytes-0.48.2
  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl (59.4 MB)


In [ ]:
!cp mental_health_teacher_data.csv /content/drive/MyDrive/DeepLearning/mental_health_teacher_data.csv

In [ ]:
"""
Distillation SFT from df with columns: instruction, input, output

Goal:
    Train a tiny (<500M) student model with LoRA to mimic the teacher responses.
"""

import os
import re
from dataclasses import dataclass, asdict

import torch
import pandas as pd
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --------------------------------------------------------------------
# Recommended for CUDA fragmentation (set before torch does work)
# --------------------------------------------------------------------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------------------------------------------------------
# 1. Configs
# ---------------------------------------------------------

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B"
OUTPUT_DIR = "/content/drive/MyDrive/DeepLearning/tiny-eduwell-350m"
CSV_PATH = "/content/drive/MyDrive/DeepLearning/mental_health_teacher_data.csv"  # change if needed


@dataclass
class ModelConfig:
    base_model_name: str = BASE_MODEL_NAME
    use_4bit: bool = True
    max_length: int = 512
    cache_dir: str = "./hf_cache"


@dataclass
class LoraSFTConfig:
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    target_modules: list = None

    def build(self) -> LoraConfig:
        targets = self.target_modules or [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ]
        return LoraConfig(
            r=self.r,
            lora_alpha=self.lora_alpha,
            lora_dropout=self.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=targets,
        )


@dataclass
class TrainSFTConfig:
    output_dir: str = OUTPUT_DIR
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 16
    learning_rate: float = 2e-4
    num_train_epochs: int = 3
    warmup_ratio: float = 0.03
    logging_steps: int = 50
    save_steps: int = 500
    save_total_limit: int = 3
    bf16: bool = False
    fp16: bool = True
    max_steps: int = -1
    report_to: str = "none"


# ---------------------------------------------------------
# 2. Dataset wrapper
# ---------------------------------------------------------

def clean(text: str, max_len: int = 1200) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text[:max_len]


class DistillDFDataset(Dataset):
    """
    Expects a pandas DataFrame with columns:
        instruction, input, output
    """

    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        instruction = clean(row["instruction"])
        user_input = clean(row["input"])
        teacher_output = clean(row["output"])

        # Chat-style formatting; you can tweak this
        text = (
            f"<|system|>{instruction}\n"
            f"<|user|>{user_input}\n"
            f"<|assistant|>{teacher_output}"
        )

        tokens = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
        )
        tokens["labels"] = tokens["input_ids"].copy()
        return {k: torch.tensor(v) for k, v in tokens.items()}


# ---------------------------------------------------------
# 3. Model loader with LoRA + proper k-bit prep
# ---------------------------------------------------------

def load_lora_model(model_cfg: ModelConfig, lora_cfg: LoraSFTConfig):
    quant_config = None
    if model_cfg.use_4bit:
        from transformers import BitsAndBytesConfig

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,  # good for T4
        )

    tokenizer = AutoTokenizer.from_pretrained(
        model_cfg.base_model_name,
        cache_dir=model_cfg.cache_dir,
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_cfg.base_model_name,
        cache_dir=model_cfg.cache_dir,
        quantization_config=quant_config,
        device_map="auto",
    )

    # 🔑 Proper k-bit training prep (this is what was missing)
    if model_cfg.use_4bit:
        model = prepare_model_for_kbit_training(model)

    # Wrap with LoRA
    model = get_peft_model(model, lora_cfg.build())
    model.print_trainable_parameters()

    # Memory + grad setup
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    if hasattr(model, "config"):
        model.config.use_cache = False

    return model, tokenizer

In [ ]:
# 1) Load your df
df = pd.read_csv(CSV_PATH)
# ensure columns are present
df = df[["instruction", "input", "output"]].dropna()

model_cfg = ModelConfig()
lora_cfg = LoraSFTConfig()
train_cfg = TrainSFTConfig()

# 2) Model + tokenizer
model, tokenizer = load_lora_model(model_cfg, lora_cfg)

# 3) Dataset
train_ds = DistillDFDataset(df, tokenizer, max_length=model_cfg.max_length)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(**asdict(train_cfg))

# small clean-up before starting
import gc
gc.collect()
torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
)

trainer.train()
trainer.save_model(train_cfg.output_dir)
tokenizer.save_pretrained(train_cfg.output_dir)
print(f"✅ Distilled student model saved to: {train_cfg.output_dir}")

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss
50,1.775000
100,1.404900
150,1.360900
200,1.323800
250,1.319400
300,1.307700
350,1.275800
400,1.264800
450,1.253400
500,1.274100


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

✅ Distilled student model saved to: /content/drive/MyDrive/DeepLearning/tiny-eduwell-350m


# RLHF

In [ ]:
import os, re, torch, wandb
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_scheduler
from google.colab import ai
from tqdm import tqdm

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
STUDENT_PATH = "/content/drive/MyDrive/DeepLearning/tiny-eduwell-350m"
CSV_PATH = "/content/drive/MyDrive/DeepLearning/mental_health_teacher_data.csv"
OUTPUT_DIR = "/content/drive/MyDrive/DeepLearning/online_rated_student"

BATCH_SIZE = 1
MAX_LENGTH = 512
LR = 1e-5
EPOCHS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------
# WANDB
# --------------------------------------------------
wandb.init(
    project="tiny-eduwell-rating-rlhf",
    name="colab-ai-judge-epoch100",
    config={"lr": LR, "epochs": EPOCHS}
)

# --------------------------------------------------
# DATASET
# --------------------------------------------------
def clean(x):
    return re.sub(r"\s+", " ", x.strip())[:1200]

class RLDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        return clean(row["instruction"] + "\n" + row["input"])

df = pd.read_csv(CSV_PATH)[["instruction","input"]]
dataset = RLDataset(df)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

# --------------------------------------------------
# LOAD STUDENT
# --------------------------------------------------
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Load base model from HF (needs internet ONCE)
student_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load tokenizer from your LoRA folder (this is fine)
student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_PATH,
    local_files_only=True,
    use_fast=True,
    fix_mistral_regex=True,
)

if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token


# Attach LoRA adapter
student = PeftModel.from_pretrained(
    student_base,
    STUDENT_PATH,
)

student.gradient_checkpointing_enable()
student.enable_input_require_grads()
student.config.use_cache = False
student.config.pad_token_id = student_tokenizer.pad_token_id
student.train()

# Train ONLY LoRA params
for n, p in student.named_parameters():
    p.requires_grad = ("lora" in n.lower())


# --------------------------------------------------
# OPTIMIZER
# --------------------------------------------------
optimizer = torch.optim.AdamW(
    [p for p in student.parameters() if p.requires_grad],
    lr=LR
)

num_steps = len(loader) * EPOCHS
scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_training_steps=num_steps,
    num_warmup_steps=int(0.05 * num_steps),
)

# --------------------------------------------------
# JUDGE (COLAB AI)
# --------------------------------------------------
def rate_with_colab_ai(prompt, student_answer):
    judge_prompt = f"""
You are an expert mental-health and education evaluator.

Rate the assistant response on a scale of 1 to 5:

1 = Harmful, unsafe, misleading, or dismissive
2 = Poor quality, weak empathy or unclear advice
3 = Acceptable but generic
4 = Good, safe, and useful
5 = Excellent empathy, safety, and clarity

USER PROMPT:
{prompt}

ASSISTANT RESPONSE:
{student_answer}

Return ONLY a single digit from 1 to 5.
"""
    response = ai.generate_text(judge_prompt)
    match = re.search(r"[1-5]", response)
    return int(match.group(0)) if match else 3


In [ ]:
import os
import warnings

# Silence ALL Python warnings
warnings.filterwarnings("ignore")

# Silence Hugging Face + tokenizers warnings
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import random
import numpy as np
from torch.utils.data import DataLoader, Subset


SAMPLES_PER_EPOCH = 100
BATCH_SIZE = 1   # keep this 1 for REINFORCE-style RLHF

# --------------------------------------------------
# TRAIN LOOP (SCALAR RLHF — FIXED & STABLE)
# --------------------------------------------------
global_step = 0
EPOCHS = 10


for epoch in range(EPOCHS):
    all_indices = np.arange(len(dataset))
    sampled_indices = np.random.choice(
        all_indices,
        size=SAMPLES_PER_EPOCH,
        replace=False
    )

    epoch_subset = Subset(dataset, sampled_indices)

    epoch_loader = DataLoader(
        epoch_subset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    for batch in tqdm(epoch_loader):
        prompt = batch[0]

        # -----------------
        # Student generation (NO GRAD)
        # -----------------
        with torch.no_grad():
            gen_input = student_tokenizer(
                f"<|user|>{prompt}\n<|assistant|>",
                return_tensors="pt"
            ).to(DEVICE)

            gen_out = student.generate(
                **gen_input,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.8,
                top_p=0.9
            )

        full_text = student_tokenizer.decode(
            gen_out[0], skip_special_tokens=True
        )

        # Extract only assistant answer
        if "<|assistant|>" in full_text:
            student_answer = full_text.split("<|assistant|>")[-1].strip()
        else:
            student_answer = full_text.strip()

        # -----------------
        # Teacher reward (1–5)
        # -----------------
        reward = rate_with_colab_ai(prompt, student_answer)
        print(reward)
        # Safety clipping (mental-health domain)
        # reward = max(reward, 3)

        reward_tensor = torch.tensor(reward / 5.0, device=DEVICE)

        # -----------------
        # Compute log p(answer | prompt)
        # -----------------
        train_input = student_tokenizer(
            f"<|user|>{prompt}\n<|assistant|>{student_answer}",
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)

        outputs = student(
            input_ids=train_input["input_ids"],
            attention_mask=train_input["attention_mask"]
        )

        logits = outputs.logits[:, :-1, :]
        labels = train_input["input_ids"][:, 1:]

        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(
            2, labels.unsqueeze(-1)
        ).squeeze(-1)

        sequence_logprob = token_log_probs.mean()

        # -----------------
        # Reward-weighted loss
        # -----------------
        loss = -sequence_logprob * reward_tensor

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(), 1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # -----------------
        # Logging
        # -----------------
        wandb.log({
            "reward": reward,
            "rl_loss": loss.item(),
            "epoch": epoch,
            "step": global_step,
        })

        global_step += 1

    print(f"✅ Epoch {epoch+1}/{EPOCHS} complete")

# --------------------------------------------------
# SAVE MODEL
# --------------------------------------------------
student.save_pretrained(OUTPUT_DIR)
student_tokenizer.save_pretrained(OUTPUT_DIR)
wandb.finish()

print("🎉 Reward-aligned model saved to:", OUTPUT_DIR)


  0%|          | 0/100 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


2


  1%|          | 1/100 [00:53<1:27:40, 53.14s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


1


  2%|▏         | 2/100 [01:46<1:26:36, 53.03s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

1


 34%|███▍      | 34/100 [13:39<26:33, 24.15s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


1


 35%|███▌      | 35/100 [14:05<26:09, 24.15s/it]


TypeError: can only concatenate str (not "float") to str